# Experiments - SKILL 2025

In [119]:
import os
import pandas as pd
from config import algorithm_strategy_pairs
from utils.simulation_utils import get_directory_for_algorithm
from scipy.stats import chi2

In [120]:
from IPython.display import display

def get_sorted_results_by_regret(arms, timestep=1000000):
    """
    Combine average results and variance calculations for given arms and timestep.

    Returns:
        pandas.DataFrame: Combined DataFrame with average and variance columns.
    """
    avg_results = {}
    var_results = {}

    for algorithm, strategy in algorithm_strategy_pairs:
        combination_name = "_".join(str(int(arm * 1000)) for arm in arms)
        results_dir = get_directory_for_algorithm(algorithm, strategy["params"]).replace('/src', '', 1)
        avg_path = os.path.join(results_dir, f'average_results_{combination_name}.csv')
        var_path = os.path.join(results_dir, f'results_{combination_name}.csv')
        strategy_suffix = "_".join(f"{k}-{v}" for k, v in strategy["params"].items()) or "default"
        results_key = f"{algorithm.name}_{strategy_suffix}"

        # Average results
        if os.path.exists(avg_path):
            df_avg = pd.read_csv(avg_path)
            filtered_avg = df_avg[df_avg['Timestep'] == timestep]
            if not filtered_avg.empty:
                avg_results[results_key] = filtered_avg.iloc[0]
        
        # Variance results
        if os.path.exists(var_path):
            df_var = pd.read_csv(var_path)
            filtered_var = df_var[df_var['Timestep'] == timestep]
            if not filtered_var.empty:
                reward_variance = filtered_var['Total Reward'].var()
                regret_variance = filtered_var['Total Regret'].var()
                avg_reward = filtered_var['Total Reward'].mean()
                avg_regret = filtered_var['Total Regret'].mean()
            else:
                reward_variance = None
                regret_variance = None
                avg_reward = None
                avg_regret = None
            var_results[results_key] = {
                "Reward Variance": reward_variance,
                "Regret Variance": regret_variance,
                "Average Reward": avg_reward,
                "Average Regret": avg_regret
            }
        else:
            var_results[results_key] = {
                "Reward Variance": None,
                "Regret Variance": None,
                "Average Reward": None,
                "Average Regret": None
            }

    # Merge results
    avg_df = pd.DataFrame.from_dict(avg_results, orient='index')
    var_df = pd.DataFrame.from_dict(var_results, orient='index')
    combined_df = avg_df.join(var_df, rsuffix='_var')

    # Remove duplicate columns (those with _var suffix)
    duplicate_cols = [col for col in combined_df.columns if col.endswith('_var')]
    combined_df = combined_df.drop(columns=duplicate_cols)

    # Remove 'Average Reward' column if present (since it's the same as 'Average Total Reward')
    if "Average Reward" in combined_df.columns:
        combined_df = combined_df.drop(columns=["Average Reward"])

    # Optional: sort by Average Regret if present
    if "Average Regret" in combined_df.columns:
        combined_df = combined_df.sort_values(by="Average Regret", ascending=False)

    display(combined_df)
    return combined_df


In [121]:
# Define a function to calculate the ratio of choosing a suboptimal arm
def calculate_suboptimal_ratio(arms, timestep):
    results = {}
    for algorithm, strategy in algorithm_strategy_pairs:
        combination_name = "_".join(str(int(arm * 1000)) for arm in arms)
        results_dir = get_directory_for_algorithm(algorithm, strategy["params"]).replace('/src', '', 1)
        detailed_results_path = os.path.join(results_dir, f'average_results_{combination_name}.csv')

        if os.path.exists(detailed_results_path):
            df = pd.read_csv(detailed_results_path)
            filtered_data = df[df['Timestep'] == timestep]
            if not filtered_data.empty:
                suboptimal_ratio = filtered_data['Average Suboptimal Arms'] / timestep
                strategy_suffix = "_".join(f"{k}-{v}" for k, v in strategy["params"].items()) or "default"
                results_key = f"{algorithm.name}_{strategy_suffix}"
                results[results_key] = suboptimal_ratio.values[0]
    return results

In [122]:
# Calculate statistical significance (p-value) for the variance of each algorithm
# Null hypothesis: observed variance is not significantly different from zero

def calculate_variance_significance(df, variance_column='Reward Variance', dof=999999, null_variance=90000):
    """
    Calculates the p-value for the observed variance for each algorithm.
    Assumes sample size (degrees of freedom) is large (default: 999999 for T=1,000,000).
    """
    significance_results = {}
    for idx, row in df.iterrows():
        observed_var = row[variance_column]
        chi2_stat = dof * observed_var / null_variance
        p_value = 1 - chi2.cdf(chi2_stat, dof)
        significance_results[idx] = {
            'Observed Variance': observed_var,
            'p-value': p_value
        }
    significance_df = pd.DataFrame.from_dict(significance_results, orient='index')
    display(significance_df.style.format({'Observed Variance': '{:,.2f}', 'p-value': '{:.2e}'}).set_caption("Variance Significance Results"))
    return significance_results

## Scenario A: Baseline

In [123]:
sorted_results_by_regret = get_sorted_results_by_regret([0.8, 0.9], timestep=1000000)

# Calculate the ratio for Timestep=10000
suboptimal_ratio_10000 = calculate_suboptimal_ratio([0.8, 0.9], 10000)
print("Suboptimal Ratio for Timestep=10000:")
print(suboptimal_ratio_10000)

# Calculate the ratio for Timestep=1000000
suboptimal_ratio_1000000 = calculate_suboptimal_ratio([0.8, 0.9], 1000000)
print("Suboptimal Ratio for Timestep=1000000:")
print(suboptimal_ratio_1000000)

variance_significance_10000 = calculate_variance_significance(sorted_results_by_regret, variance_column='Reward Variance', dof=999999, null_variance=90000)

,Timestep,Average Total Reward,Average Suboptimal Arms,Average Regret,Average Zeros Count,Average Ones Count,Reward Variance,Regret Variance
UCB-Improved_delta-1,1000000.0,867513.01,325001.06,32500.106,132486.99,867513.01,2.143279e+09,2.140127e+09
Greedy_epsilon-0.5,1000000.0,875020.64,250022.03,25002.203,124979.36,875020.64,9.949043e+04,1.514403e+03
ETC_exploration_rounds-100000,1000000.0,889997.60,100000.00,10000.000,110002.40,889997.60,8.343869e+04,0.000000e+00
ETC_exploration_rounds-10,1000000.0,893010.47,70040.82,7004.082,106989.53,893010.47,6.586476e+08,6.575005e+08
Greedy_epsilon-0.1,1000000.0,895001.31,50084.76,5008.476,104998.69,895001.31,8.004878e+04,7.615663e+02
Greedy_epsilon-0.05,1000000.0,897483.41,25184.69,2518.469,102516.59,897483.41,8.456372e+04,1.355023e+03
ETC_exploration_rounds-10000,1000000.0,899003.34,10000.00,1000.000,100996.66,899003.34,7.795299e+04,0.000000e+00
Greedy_epsilon-0.01,1000000.0,899422.38,5880.14,588.014,100577.62,899422.38,1.073638e+05,1.514006e+04
Greedy_epsilon-0.005,1000000.0,899581.43,4229.88,422.988,100418.57,899581.43,1.066075e+05,4.913531e+04
UCB_default,1000000.0,899766.19,2381.75,238.175,100233.81,899766.19,7.851543e+04,1.306600e+03


Suboptimal Ratio for Timestep=10000:
{'ETC_exploration_rounds-10': 0.07408200000000001, 'ETC_exploration_rounds-100': 0.010095, 'ETC_exploration_rounds-1000': 0.1, 'ETC_exploration_rounds-10000': 0.5, 'ETC_exploration_rounds-100000': 0.5, 'Greedy_epsilon-0.005': 0.17617, 'Greedy_epsilon-0.01': 0.093266, 'Greedy_epsilon-0.05': 0.042832999999999996, 'Greedy_epsilon-0.1': 0.059226999999999995, 'Greedy_epsilon-0.5': 0.250793, 'UCB_default': 0.08526900000000001, 'UCB-Tuned_default': 0.01659, 'UCB-V_theta-1_c-1_b-1': 0.060917, 'PAC-UCB_c-1_b-1_q-1.3_beta-0.05': 0.07373400000000001, 'UCB-Improved_delta-1': 0.325106, 'EUCBV_rho-0.5': 0.034651999999999995}
Suboptimal Ratio for Timestep=1000000:
{'ETC_exploration_rounds-10': 0.07004082, 'ETC_exploration_rounds-100': 0.00010095, 'ETC_exploration_rounds-1000': 0.001, 'ETC_exploration_rounds-10000': 0.01, 'ETC_exploration_rounds-100000': 0.1, 'Greedy_epsilon-0.005': 0.00422988, 'Greedy_epsilon-0.01': 0.005880140000000001, 'Greedy_epsilon-0.05': 0.0

,Observed Variance,p-value
UCB-Improved_delta-1,"2,143,278,875.08",0.00e+00
Greedy_epsilon-0.5,"99,490.43",0.00e+00
ETC_exploration_rounds-100000,"83,438.69",1.00e+00
ETC_exploration_rounds-10,"658,647,565.52",0.00e+00
Greedy_epsilon-0.1,"80,048.78",1.00e+00
Greedy_epsilon-0.05,"84,563.72",1.00e+00
ETC_exploration_rounds-10000,"77,952.99",1.00e+00
Greedy_epsilon-0.01,"107,363.81",0.00e+00
Greedy_epsilon-0.005,"106,607.54",0.00e+00
UCB_default,"78,515.43",1.00e+00


## Scenario B: Low-Variance Micro-Gap

In [124]:
sorted_results_by_regret = get_sorted_results_by_regret([0.895, 0.9], timestep=1000000)

# Calculate the ratio for Timestep=10000
suboptimal_ratio_10000 = calculate_suboptimal_ratio([0.895, 0.9], 10000)
print("Suboptimal Ratio for Timestep=10000:")
print(suboptimal_ratio_10000)

# Calculate the ratio for Timestep=1000000
suboptimal_ratio_1000000 = calculate_suboptimal_ratio([0.895, 0.9], 1000000)
print("Suboptimal Ratio for Timestep=1000000:")
print(suboptimal_ratio_1000000)

variance_significance_10000 = calculate_variance_significance(sorted_results_by_regret, variance_column='Reward Variance', dof=999999, null_variance=90000)

,Timestep,Average Total Reward,Average Suboptimal Arms,Average Regret,Average Zeros Count,Average Ones Count,Reward Variance,Regret Variance
UCB-Improved_delta-1,1000000.0,897476.88,505000.00,2525.0026,102523.12,897476.88,6.194415e+06,5.744238e+06
ETC_exploration_rounds-10,1000000.0,897848.97,430160.03,2150.8011,102151.03,897848.97,6.432726e+06,6.185366e+06
ETC_exploration_rounds-100,1000000.0,897945.33,410103.96,2050.5205,102054.67,897945.33,6.280732e+06,6.103198e+06
Greedy_epsilon-0.005,1000000.0,898574.61,285740.62,1428.7037,101425.39,898574.61,3.665458e+06,3.543181e+06
ETC_exploration_rounds-1000,1000000.0,898712.26,258225.42,1291.1267,101287.74,898712.26,4.837382e+06,4.719747e+06
Greedy_epsilon-0.5,1000000.0,898737.76,252903.26,1264.5185,101262.24,898737.76,7.755271e+04,7.140372e+02
UCB_default,1000000.0,898880.02,225434.51,1127.1738,101119.98,898880.02,9.187283e+04,1.597644e+04
Greedy_epsilon-0.01,1000000.0,898907.05,220636.24,1103.1807,101092.95,898907.05,2.434165e+06,2.472325e+06
ETC_exploration_rounds-100000,1000000.0,899508.90,100000.00,500.0000,100491.10,899508.90,7.935656e+04,0.000000e+00
EUCBV_rho-0.5,1000000.0,899546.35,92272.12,461.3585,100453.65,899546.35,1.091429e+05,1.682131e+04


Suboptimal Ratio for Timestep=10000:
{'ETC_exploration_rounds-10': 0.44323900000000005, 'ETC_exploration_rounds-100': 0.42118599999999995, 'ETC_exploration_rounds-1000': 0.345947, 'ETC_exploration_rounds-10000': 0.5, 'ETC_exploration_rounds-100000': 0.5, 'Greedy_epsilon-0.005': 0.773701, 'Greedy_epsilon-0.01': 0.687346, 'Greedy_epsilon-0.05': 0.441722, 'Greedy_epsilon-0.1': 0.45015900000000003, 'Greedy_epsilon-0.5': 0.373055, 'UCB_default': 0.463031, 'UCB-Tuned_default': 0.397223, 'UCB-V_theta-1_c-1_b-1': 0.427695, 'PAC-UCB_c-1_b-1_q-1.3_beta-0.05': 0.43619300000000005, 'UCB-Improved_delta-1': 0.505, 'EUCBV_rho-0.5': 0.41811000000000004}
Suboptimal Ratio for Timestep=1000000:
{'ETC_exploration_rounds-10': 0.43016003, 'ETC_exploration_rounds-100': 0.41010396000000005, 'ETC_exploration_rounds-1000': 0.25822542000000004, 'ETC_exploration_rounds-10000': 0.03013264, 'ETC_exploration_rounds-100000': 0.1, 'Greedy_epsilon-0.005': 0.28574062, 'Greedy_epsilon-0.01': 0.22063623999999998, 'Greedy_

,Observed Variance,p-value
UCB-Improved_delta-1,"6,194,415.18",0.00e+00
ETC_exploration_rounds-10,"6,432,726.05",0.00e+00
ETC_exploration_rounds-100,"6,280,731.62",0.00e+00
Greedy_epsilon-0.005,"3,665,457.69",0.00e+00
ETC_exploration_rounds-1000,"4,837,381.73",0.00e+00
Greedy_epsilon-0.5,"77,552.71",1.00e+00
UCB_default,"91,872.83",0.00e+00
Greedy_epsilon-0.01,"2,434,165.12",0.00e+00
ETC_exploration_rounds-100000,"79,356.56",1.00e+00
EUCBV_rho-0.5,"109,142.90",0.00e+00


## Scenario C: High-Variance Micro-Gap

In [125]:
sorted_results_by_regret = get_sorted_results_by_regret([0.89, 0.895], timestep=1000000)

# Calculate the ratio for Timestep=10000
suboptimal_ratio_10000 = calculate_suboptimal_ratio([0.89, 0.895], 10000)
print("Suboptimal Ratio for Timestep=10000:")
print(suboptimal_ratio_10000)

# Calculate the ratio for Timestep=1000000
suboptimal_ratio_1000000 = calculate_suboptimal_ratio([0.89, 0.895], 1000000)
print("Suboptimal Ratio for Timestep=1000000:")
print(suboptimal_ratio_1000000)

variance_significance_10000 = calculate_variance_significance(sorted_results_by_regret, variance_column='Reward Variance', dof=999999, null_variance=93975)

,Timestep,Average Total Reward,Average Suboptimal Arms,Average Regret,Average Zeros Count,Average Ones Count,Reward Variance,Regret Variance
UCB-Improved_delta-1,1000000.0,892543.32,490000.14,2450.0033,107456.68,892543.32,6.169246e+06,5.805479e+06
ETC_exploration_rounds-10,1000000.0,892846.03,429999.16,2149.9970,107153.97,892846.03,6.453856e+06,6.188592e+06
ETC_exploration_rounds-100,1000000.0,892991.64,400123.02,2000.6160,107008.36,892991.64,6.217678e+06,6.055667e+06
Greedy_epsilon-0.005,1000000.0,893225.36,352517.89,1762.5908,106774.64,893225.36,4.012222e+06,3.848576e+06
Greedy_epsilon-0.5,1000000.0,893722.15,253809.30,1269.0495,106277.85,893722.15,8.282439e+04,7.740712e+02
Greedy_epsilon-0.01,1000000.0,893787.47,240771.05,1203.8550,106212.53,893787.47,2.644339e+06,2.519986e+06
UCB_default,1000000.0,893819.56,234514.58,1172.5742,106180.44,893819.56,9.387380e+04,2.249400e+04
ETC_exploration_rounds-1000,1000000.0,893831.50,231292.99,1156.4650,106168.50,893831.50,4.469433e+06,4.441563e+06
ETC_exploration_rounds-100000,1000000.0,894493.54,100000.00,500.0000,105506.46,894493.54,7.776199e+04,0.000000e+00
EUCBV_rho-0.5,1000000.0,894514.12,95262.97,476.3120,105485.88,894514.12,9.623397e+04,2.029088e+04


Suboptimal Ratio for Timestep=10000:
{'ETC_exploration_rounds-10': 0.42991899999999994, 'ETC_exploration_rounds-100': 0.411488, 'ETC_exploration_rounds-1000': 0.329764, 'ETC_exploration_rounds-10000': 0.5, 'ETC_exploration_rounds-100000': 0.5, 'Greedy_epsilon-0.005': 0.7770130000000001, 'Greedy_epsilon-0.01': 0.647366, 'Greedy_epsilon-0.05': 0.40628400000000003, 'Greedy_epsilon-0.1': 0.442817, 'Greedy_epsilon-0.5': 0.416001, 'UCB_default': 0.457784, 'UCB-Tuned_default': 0.409105, 'UCB-V_theta-1_c-1_b-1': 0.42221800000000004, 'PAC-UCB_c-1_b-1_q-1.3_beta-0.05': 0.441601, 'UCB-Improved_delta-1': 0.490014, 'EUCBV_rho-0.5': 0.414894}
Suboptimal Ratio for Timestep=1000000:
{'ETC_exploration_rounds-10': 0.42999916, 'ETC_exploration_rounds-100': 0.40012302, 'ETC_exploration_rounds-1000': 0.23129299, 'ETC_exploration_rounds-10000': 0.034106910000000004, 'ETC_exploration_rounds-100000': 0.1, 'Greedy_epsilon-0.005': 0.35251789, 'Greedy_epsilon-0.01': 0.24077105, 'Greedy_epsilon-0.05': 0.06031019,

,Observed Variance,p-value
UCB-Improved_delta-1,"6,169,245.59",0.00e+00
ETC_exploration_rounds-10,"6,453,855.77",0.00e+00
ETC_exploration_rounds-100,"6,217,677.79",0.00e+00
Greedy_epsilon-0.005,"4,012,222.43",0.00e+00
Greedy_epsilon-0.5,"82,824.39",1.00e+00
Greedy_epsilon-0.01,"2,644,339.00",0.00e+00
UCB_default,"93,873.80",7.77e-01
ETC_exploration_rounds-1000,"4,469,433.28",0.00e+00
ETC_exploration_rounds-100000,"77,761.99",1.00e+00
EUCBV_rho-0.5,"96,233.97",0.00e+00
